# KKBox churn training in Google Colab

This short notebook downloads the KKBox tables, prepares them into one row per subscriber with transaction, member, listening, and date features, then trains the existing calibrated churn model.

Place these files in one Google Drive folder: `train_v2.csv`, `members_v3.csv`, `transactions_v2.csv`, and `user_logs_v2.csv`. The loader also accepts the non-versioned filenames.

In [ ]:
!pip -q install pandas scikit-learn mlflow joblib kagglehub

# Colab starts in /content, so clone the repository before importing src.
!git clone -q --branch add-KKBox-dataset https://github.com/INSAgroup15/Trustworthy-customer-churn-prediction.git /content/tccp
%cd /content/tccp
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import kagglehub

# Download the public KKBox dataset from Kaggle. The download is large.
downloaded = Path(kagglehub.dataset_download('qmdo97/kkboxdataset'))
csv_files = list(downloaded.rglob('*.csv'))
KKBOX_DIR = next((p.parent for p in csv_files if p.name in {'train.csv', 'train_v2.csv'}), downloaded)
OUTPUT = Path('/content/kkbox_features.csv')
print('Dataset folder:', KKBOX_DIR)
print('CSV files:', [p.name for p in csv_files])

In [ ]:
# Run this from the repository root.
!python -m src.kkbox_features --data-dir "{KKBOX_DIR}" --output "{OUTPUT}" --chunksize 250000

import pandas as pd
features = pd.read_csv(OUTPUT, nrows=5)
print('Generated columns:', len(features.columns))
display(features.head())

In [ ]:
# Train the calibrated baseline model and save it.
!python -m src.train --data "{OUTPUT}" --output /content/kkbox_model.joblib --mlflow-experiment kkbox-churn

In [ ]:
import joblib
from sklearn.metrics import classification_report, roc_auc_score, brier_score_loss

bundle = joblib.load('/content/kkbox_model.joblib')
probability = bundle['model'].predict_proba(bundle['X_test'])[:, 1]
prediction = (probability >= 0.50).astype(int)
print('ROC-AUC:', round(roc_auc_score(bundle['y_test'], probability), 4))
print('Brier score:', round(brier_score_loss(bundle['y_test'], probability), 4))
print(classification_report(bundle['y_test'], prediction, digits=4))